# Clase 01 — Qué es el NLP

**Curso:** NLP Clásico con Python · **Bloque 1:** Del texto a los tokens
**Tiempo estimado:** 90 minutos · **Capítulo del manual:** `manual/01_que_es_el_nlp.md`

## Qué vas a ver en esta clase

- Las **tareas típicas del NLP** y en qué clase del curso se aborda cada una.
- La **pipeline clásica**, de la adquisición de los textos a la evaluación, con una pipeline de juguete que falla y se corrige.
- **Por qué el lenguaje es difícil**: ambigüedad léxica y sintáctica (el resto de dificultades, en la clase 2).
- La **morfología rica** del italiano y del español: un experimento que genera y cuenta las formas de un verbo, los clíticos y el género y número.
- Enfoque **clásico frente a neuronal**.
- Un experimento guiado **«reglas frente a recuentos»** con un corpus pequeñísimo en italiano y en español.

## Requisitos

- Python básico (funciones, listas, diccionarios) y nociones de clasificación (curso de Machine Learning).
- Librerías: `spacy` (con `es_core_news_sm` e `it_core_news_sm`), `pandas`, `matplotlib`. La celda siguiente las instala solo si faltan (funciona igual en Google Colab).

> **Cómo usar este notebook:** ejecuta las celdas en orden. Es la clase más ligera del curso: poco código y muchos ejemplos. Todos los textos son cortos y están escritos para el curso (*ejemplo ilustrativo*). Lo que aquí solo se muestra como avance se estudia con calma en la clase que se indica.

---
## Instalación y modelos

Esta celda instala lo que falte y descarga los dos modelos de spaCy. En un entorno con todo instalado no hace nada. Si estás en Colab puede tardar un minuto la primera vez.

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("spacy", "spacy")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")

import spacy

def cargar_modelo(nombre):
    """Carga un modelo de spaCy y lo descarga si no está instalado."""
    try:
        return spacy.load(nombre)
    except OSError:
        subprocess.run([sys.executable, "-m", "spacy", "download", nombre, "-q"], check=True)
        return spacy.load(nombre)

nlp_es = cargar_modelo("es_core_news_sm")
nlp_it = cargar_modelo("it_core_news_sm")
print("Modelos de spaCy listos.")

---
## Celda 0 — Setup

Importamos las librerías y fijamos la **semilla** de los generadores aleatorios. En esta clase casi no hay azar, pero mantenemos el hábito: cada estudiante debe obtener los mismos resultados.

In [ ]:
import random
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

random.seed(42)                          # semilla fija: todos obtenemos los mismos resultados
np.random.seed(42)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)
print("Librerías cargadas correctamente.")

---
## 1. Las tareas del NLP

Casi todo problema de NLP se puede escribir como **entrada → salida**. Aquí tienes las seis familias de tareas del curso con un ejemplo real en cada idioma y la clase donde se trabaja. La **generación moderna** y la **traducción automática** quedan fuera del curso: los cursos de IA generativa son el sitio donde seguir.

In [ ]:
# datos-es
EJEMPLOS = {
    "clf": ("Mi paquete no ha llegado y nadie responde", "Il mio pacco non è arrivato e nessuno risponde"),
    "ext": ("El 3 de marzo pagamos 1.250 euros a Talleres Ruiz", "Il 3 marzo abbiamo pagato 1.250 euro all'officina Rossi"),
    "s2s": ("La reunión se ha aplazado al jueves", "La riunione è stata rinviata a giovedì"),
    "gen": ("Estimado cliente, lamentamos el retraso de su pedido y…", "Gentile cliente, ci scusiamo per il ritardo del suo ordine e…"),
    "qa": ("¿Cuál es el plazo para devolver un producto?", "Qual è il termine per restituire un prodotto?"),
    "ling": ("Los niños comieron manzanas rojas", "I bambini hanno mangiato mele rosse"),
}

In [ ]:
TAREAS = {
    "clf":  ("Clasificación", "una etiqueta (categoría de ticket, sentimiento)", "clases 5 y 6"),
    "ext":  ("Extracción", "datos concretos (fecha, importe, organización)", "clase 7 (y regex en la 2)"),
    "s2s":  ("Secuencia a secuencia", "otro texto (traducción o resumen)", "clase 11 (resumen); traducción: fuera del curso"),
    "gen":  ("Generación", "texto nuevo a partir de un comienzo", "clase 8 (idea con n-gramas); moderna: fuera del curso"),
    "qa":   ("Recuperación y respuesta", "los documentos que contienen la respuesta", "clase 11"),
    "ling": ("Análisis lingüístico", "categorías gramaticales y sintaxis", "clases 3 y 7"),
}

filas = []
for clave, (tarea, salida, donde) in TAREAS.items():
    ej_es, ej_it = EJEMPLOS[clave]
    filas.append({"tarea": tarea, "ejemplo (español)": ej_es, "ejemplo (italiano)": ej_it,
                  "qué sale": salida, "dónde": donde})
tabla_tareas = pd.DataFrame(filas)
print(tabla_tareas.to_string(index=False))

**¿Qué observas?** Cada fila se lee como «entra un texto, sale X». Fíjate en que la misma tarea existe igual en los dos idiomas, pero las herramientas (listas de stopwords, modelos de spaCy, léxicos) son distintas para cada uno. Y dos de las seis filas (traducción y generación moderna) no se estudian en este curso: saber dónde acaba lo que se cubre también forma parte del mapa.

---
## 2. La pipeline clásica

Seis pasos: **adquisición → limpieza y normalización → tokenización → representación → modelo → evaluación y uso**. Cada uno tiene su clase en el curso (2 y 3 para los pasos 2 y 3; 4 y 10 para la representación; 5, 6, 8 y 9 para el modelo; 5, 6, 7, 11 y 12 para evaluar y usar). Los pasos **se eligen en función de la tarea** y **rara vez se recorren una sola vez**: al evaluar se vuelve atrás.

Aquí tienes una pipeline de juguete que decide si un ticket es urgente. Cada paso es una función de una línea; **la tokenización con `split()` es deliberadamente ingenua** y la corregiremos en la clase 2.

In [ ]:
# datos-es
TICKETS = {"spa": "Mi pedido no ha llegado y necesito ayuda urgente.", "ita": "Il mio ordine non è arrivato e ho bisogno di aiuto urgente."}
PALABRAS_URGENTES = {"urgente", "urgentemente", "inmediatamente", "subito", "immediatamente"}

In [ ]:
def adquirir(idioma):
    return TICKETS[idioma]                                   # paso 1: aquí, un texto ya escrito

def limpiar(texto):
    return re.sub(r"\s+", " ", texto).strip().lower()         # paso 2

def tokenizar_ingenuo(texto):
    return texto.split()                                     # paso 3 (ingenuo)

def representar(tokens):
    return Counter(tokens)                                   # paso 4: una bolsa de palabras

def modelo(bolsa):
    return "urgente" if any(p in bolsa for p in PALABRAS_URGENTES) else "normal"   # paso 5: una regla

for idioma in ("spa", "ita"):
    texto = adquirir(idioma)
    tokens = tokenizar_ingenuo(limpiar(texto))
    print(idioma, len(tokens), tokens[-3:], "→", modelo(representar(tokens)))

**¿Qué observas?** El último token es `urgente.` (con el punto pegado) y la regla no lo reconoce: el ticket sale «normal» sin ningún error visible. El fallo está en el paso 3, no en el modelo. Vamos a volver atrás y cambiar solo ese paso.

In [ ]:
def tokenizar_regex(texto):
    return re.findall(r"\w+", texto)                         # el punto ya no se queda pegado

for idioma in ("spa", "ita"):
    texto = adquirir(idioma)
    tokens = tokenizar_regex(limpiar(texto))
    print(idioma, len(tokens), tokens[-3:], "→", modelo(representar(tokens)))

Con el cambio, los dos tickets salen «urgente». Esto es lo que significa que la pipeline **no es lineal**: un error detectado al final obliga a volver a un paso anterior. En la clase 2 verás qué más falla con `split()` y con `\w+` (apóstrofos del italiano, cifras, contracciones).

---
## 3. Por qué el lenguaje es difícil

### 3.1 Ambigüedad léxica

Una misma cadena tiene varios significados. Para una bolsa de palabras (clase 4) las dos apariciones de `banco` son **el mismo token**.

In [ ]:
LEXICA = [
    ("banco", "Me senté en el banco del parque", "asiento"),          # dato-es
    ("banco", "Ingresé el dinero en el banco", "entidad financiera"),          # dato-es
    ("vino", "Con la cena tomamos vino tinto", "bebida"),          # dato-es
    ("vino", "Ayer vino mi hermana a verme", "forma del verbo venir"),          # dato-es
    ("pesca", "Ho mangiato una pesca matura", "fruta"),          # dato-es
    ("pesca", "La pesca è vietata nel lago", "actividad"),          # dato-es
    ("calcio", "Il calcio è il suo sport preferito", "deporte"),          # dato-es
    ("calcio", "Il latte contiene calcio", "elemento químico"),          # dato-es
]

for palabra, frase, sentido in LEXICA:
    print(f"{palabra:>6} | {sentido:<22} | {frase}")

fr_a, fr_b = LEXICA[0][1], LEXICA[1][1]
print("¿El token 'banco' es idéntico en las dos frases?", "banco" in fr_a.split() and "banco" in fr_b.split())

**¿Qué observas?** El programa ve exactamente la misma cadena `banco` en las dos frases; solo las palabras de alrededor las distinguen. Por eso los n-gramas (clase 8) y los *embeddings* (clase 10) miran el contexto.

### 3.2 Ambigüedad sintáctica

`Vi al hombre con el telescopio` / `Ho visto l'uomo con il cannocchiale`: dos lecturas, según a qué palabra se una el grupo «con el telescopio». Como **avance** de la clase 7, mostramos los componentes de la *pipeline* de spaCy y a qué palabra une el analizador ese grupo.

In [ ]:
# datos-es
FRASES_SINTAXIS = [(nlp_es, "Vi al hombre con el telescopio", "telescopio"), (nlp_it, "Ho visto l'uomo con il cannocchiale", "cannocchiale")]

In [ ]:
for nlp, frase, palabra_clave in FRASES_SINTAXIS:
    doc = nlp(frase)
    objetivo = [t for t in doc if t.text == palabra_clave][0]
    print(nlp.meta["lang"], "|", nlp.pipe_names)
    print("   el grupo se une a:", objetivo.head.text, "(", objetivo.dep_, ")")

**¿Qué observas?** La *pipeline* tiene varios componentes (`tok2vec`, `morphologizer`, `parser`, `lemmatizer`, `ner`…) y el analizador **elige** una lectura. Recuerda: **las dos lecturas son válidas**; el modelo elige una porque ha visto muchas frases parecidas, no porque «entienda» la escena. Por eso esta cifra no se afirma en ningún test del curso: con otra versión del modelo puede cambiar.

> **Nota:** el resto de dificultades (ambigüedad referencial, contexto y pragmática, variación del texto real) se estudia en la clase 2, donde empieza el trabajo con texto bruto.

---
## 4. La morfología rica del italiano y del español

### 4.1 Experimento guiado: ¿cuántas formas tiene un verbo?

Generamos con código los **tiempos simples regulares** del verbo `cantar` (español) y `cantare` (italiano) para las seis personas, más las formas no personales (infinitivo, gerundio y participio en cuatro formas), y contamos las **formas distintas**. Después las comparamos con las cinco formas de un verbo inglés (`sing`, `sings`, `sang`, `sung`, `singing`).

In [ ]:
# datos-es
TERMINACIONES_ES = {
    "presente": ["o", "as", "a", "amos", "áis", "an"],
    "imperfecto": ["aba", "abas", "aba", "ábamos", "abais", "aban"],
    "indefinido": ["é", "aste", "ó", "amos", "asteis", "aron"],
    "futuro": ["aré", "arás", "ará", "aremos", "aréis", "arán"],
    "condicional": ["aría", "arías", "aría", "aríamos", "aríais", "arían"],
    "subjuntivo": ["e", "es", "e", "emos", "éis", "en"],
}
TERMINACIONES_IT = {
    "presente": ["o", "i", "a", "iamo", "ate", "ano"],
    "imperfetto": ["avo", "avi", "ava", "avamo", "avate", "avano"],
    "remoto": ["ai", "asti", "ò", "ammo", "aste", "arono"],
    "futuro": ["erò", "erai", "erà", "eremo", "erete", "eranno"],
    "condizionale": ["erei", "eresti", "erebbe", "eremmo", "ereste", "erebbero"],
    "congiuntivo": ["i", "i", "i", "iamo", "iate", "ino"],
}
NO_PERSONALES_ES = ["cantar", "cantando", "cantado", "cantada", "cantados", "cantadas"]
NO_PERSONALES_IT = ["cantare", "cantando", "cantato", "cantata", "cantati", "cantate"]
FORMAS_INGLES = ["sing", "sings", "sang", "sung", "singing"]

In [ ]:
def conjugar(raiz, terminaciones):
    """Devuelve {tiempo: [6 formas]} añadiendo cada terminación a la raíz."""
    return {tiempo: [raiz + t for t in lista] for tiempo, lista in terminaciones.items()}

formas_es = conjugar("cant", TERMINACIONES_ES)
formas_it = conjugar("cant", TERMINACIONES_IT)

for nombre, formas in (("Español", formas_es), ("Italiano", formas_it)):
    print(nombre)
    for tiempo, lista in formas.items():
        print(f"  {tiempo:<13}", ", ".join(lista))

In [ ]:
def contar_formas(formas_personales, no_personales):
    lista = [f for lista in formas_personales.values() for f in lista] + no_personales
    return len(lista), len(set(lista)), lista

huecos_es, distintas_es, lista_es = contar_formas(formas_es, NO_PERSONALES_ES)
huecos_it, distintas_it, lista_it = contar_formas(formas_it, NO_PERSONALES_IT)

print(f"Español: {huecos_es} huecos, {distintas_es} formas distintas")
print(f"Italiano: {huecos_it} huecos, {distintas_it} formas distintas")
print(f"Inglés (sing): {len(FORMAS_INGLES)} formas distintas")
print(f"Veces más formas que en inglés: {distintas_es / len(FORMAS_INGLES):.1f} (es) y {distintas_it / len(FORMAS_INGLES):.1f} (it)")

repetidas_es = sorted(f for f, n in Counter(lista_es).items() if n > 1)
repetidas_it = sorted(f for f, n in Counter(lista_it).items() if n > 1)
print("Formas que coinciden en más de un hueco (es):", repetidas_es)
print("Formas que coinciden en más de un hueco (it):", repetidas_it)

In [ ]:
etiquetas = ["español", "italiano", "inglés"]
valores = [distintas_es, distintas_it, len(FORMAS_INGLES)]
plt.bar(etiquetas, valores, color=["#c0392b", "#27ae60", "#2980b9"])
for i, v in enumerate(valores):
    plt.text(i, v + 0.5, str(v), ha="center")
plt.title("Formas distintas de un verbo regular (ejemplo ilustrativo)")
plt.ylabel("Formas distintas")
plt.show()

**¿Qué observas?** Un verbo regular en -ar / -are se escribe de **38** (español) y **37** (italiano) formas distintas con estos tiempos, frente a **5** del verbo inglés: entre siete y ocho veces más. El número de formas distintas es menor que el de huecos (42) porque algunas coinciden: `cantamos` es a la vez presente e indefinido; en italiano `canti` sirve para tres personas del congiuntivo y una del presente. Y son solo los tiempos simples de un verbo **regular**. Para una bolsa de palabras, cada forma es un rasgo distinto: con pocos datos, esa dispersión duele.

Ahora, un experimento **abierto**: pasamos cada forma aislada por el lematizador de spaCy y contamos cuántas vuelven al infinitivo. Solo se muestra el resultado, no se afirma.

In [ ]:
for nombre, nlp, lista, infinitivo in (("Español", nlp_es, lista_es, "cantar"), ("Italiano", nlp_it, lista_it, "cantare")):
    unicas = sorted(set(lista))
    lemas = {f: nlp(f)[0].lemma_ for f in unicas}
    vuelven = [f for f, lema in lemas.items() if lema == infinitivo]
    fallan = [(f, lemas[f]) for f in unicas if lemas[f] != infinitivo][:6]
    print(f"{nombre}: {len(vuelven)} de {len(unicas)} formas vuelven al infinitivo")
    print("   Ejemplos que no vuelven:", fallan)

**¿Qué observas?** El lematizador (con spaCy 3.8 y los modelos 3.8.0; puede variar ligeramente con otras versiones) devuelve el infinitivo solo para una parte de las formas, y a veces inventa lemas (`cantabai`). Una forma suelta no da contexto: el lematizador depende de la categoría gramatical, que depende de la frase. Lo veremos en la clase 3.

### 4.2 Clíticos

Los pronombres átonos se pegan al verbo: `dámelo`, `dammelo`, `dándoselo`, `dandoglielo`. Una sola cadena contiene un verbo y dos pronombres.

In [ ]:
# datos-es
CLITICOS = [(nlp_es, "Dámelo ahora"), (nlp_es, "Quiero dárselo mañana"), (nlp_it, "Dammelo adesso"), (nlp_it, "Voglio darglielo domani")]

In [ ]:
for nlp, frase in CLITICOS:
    doc = nlp(frase)
    print(f"{frase!r}: split() → {len(frase.split())} palabras;", "spaCy →", [(t.text, t.lemma_) for t in doc])

**¿Qué observas?** Ni `split()` ni el tokenizador de spaCy separan `dámelo` en `da` + `me` + `lo`, y el lema que devuelve el modelo para las formas con clítico no es el verbo (con estos modelos; puede variar con otras versiones). Es un problema que aparece en las clases 2 y 3.

### 4.3 Género y número

Los adjetivos concuerdan con el sustantivo: cuatro formas en español e italiano, una en inglés.

In [ ]:
adjetivos = {"español": ["rojo", "roja", "rojos", "rojas"], "italiano": ["rosso", "rossa", "rossi", "rosse"], "inglés": ["red"]}          # dato-es
for idioma, formas in adjetivos.items():
    print(f"{idioma:<9} {len(set(formas))} forma(s): {formas}")

for nlp, frase in ((nlp_es, "Los coches rojos son bonitos"), (nlp_it, "Le macchine rosse sono belle")):          # dato-es
    print([(t.text, t.lemma_) for t in nlp(frase)])

**¿Qué observas?** En una frase completa, el lematizador sí devuelve el lema masculino singular (`rojo`, `rosso`, `bonito`, `bello`): con contexto funciona mucho mejor que con formas sueltas. Aun así, hay que revisarlo en cada idioma. **Regla práctica:** las herramientas pensadas para inglés no se trasladan sin adaptar (stopwords, lematizadores, acentos, apóstrofos).

---
## 5. Enfoque clásico frente a enfoque neuronal

Ninguno es siempre mejor. Esta tabla es honesta sobre cuándo conviene cada uno; en la **clase 12** los compararás con cifras sobre un mismo problema.

| Criterio | Clásico | Neuronal (transformer) |
|---|---|---|
| Datos necesarios | Pocos: cientos de ejemplos suelen bastar | Preentrenado sabe mucho; ajustarlo bien pide más ejemplos |
| Coste de cálculo | Bajo (un portátil, segundos) | Alto (memoria, tiempo, a menudo GPU) |
| Explicabilidad | Alta: se leen reglas y pesos | Baja: cuesta decir por qué eligió una etiqueta |
| Robustez (variación, contexto) | Frágil ante ortografía, negación, ironía | Mucho mejor con contexto y variación |
| Idiomas con pocos recursos | Depende de recursos por idioma | Multilingüe ayuda, con sesgo hacia idiomas grandes |
| Mantenimiento | Sencillo de depurar y actualizar | Más costoso (versiones, despliegue, cambios de comportamiento) |

**Conviene lo clásico** cuando hay pocos datos etiquetados, hace falta explicar cada decisión, el presupuesto de cálculo es mínimo o una expresión regular ya resuelve el problema. **Conviene lo neuronal** cuando el contexto y la variación pesan mucho y hay recursos para usarlo.

---
## 6. Experimento guiado: reglas frente a recuentos

Un mini-problema de **sentimiento** con un corpus pequeñísimo escrito para el curso (**ejemplo ilustrativo**): 12 frases de entrenamiento y 8 de prueba por idioma, equilibradas (6 + 6 para entrenar, 4 + 4 para probar) y con **dos casos de negación** en cada conjunto de prueba. Solo Python puro, sin scikit-learn.

Tres clasificadores: (1) **reglas**: listas de palabras positivas y negativas; (2) **reglas con negación**: un negador invierte la polaridad de la palabra que aparece en las tres posiciones siguientes; (3) **recuentos**: un `Counter` de palabras por clase, contado en el entrenamiento. Si hay empate o ninguna palabra conocida, el clasificador **no decide** (`None`) y cuenta como error.

> **Atención:** con 8 frases de prueba, un solo acierto de diferencia mueve 12,5 puntos. Los porcentajes son **ilustrativos, no generalizables**.

In [ ]:
# datos-es
ENTRENAMIENTO_ES = [
    ("el producto es excelente y muy útil", "pos"),
    ("me encantó el servicio, todo perfecto", "pos"),
    ("la entrega fue rápida y el envío llegó bien", "pos"),
    ("muy buena calidad, estoy contento con la compra", "pos"),
    ("atención amable y precio justo", "pos"),
    ("una experiencia fantástica, lo recomiendo", "pos"),
    ("el producto es malo y muy caro", "neg"),
    ("odié el servicio, todo un desastre", "neg"),
    ("la entrega fue lenta y el envío llegó roto", "neg"),
    ("pésima calidad, estoy decepcionado con la compra", "neg"),
    ("atención horrible y precio abusivo", "neg"),
    ("una experiencia terrible, lo desaconsejo", "neg"),
]
PRUEBA_ES = [
    ("un producto excelente, todo perfecto", "pos"),
    ("la atención fue amable y el envío rápido", "pos"),
    ("el envío no fue lento", "pos"),
    ("una compra genial", "pos"),
    ("un desastre, todo muy caro", "neg"),
    ("la calidad no es buena", "neg"),
    ("una compra lamentable", "neg"),
    ("¡qué maravilla, se rompió al primer día!", "neg"),
]
ENTRENAMIENTO_IT = [
    ("il prodotto è eccellente e molto utile", "pos"),
    ("mi è piaciuto il servizio, tutto perfetto", "pos"),
    ("la consegna è stata veloce e la spedizione è arrivata bene", "pos"),
    ("ottima qualità, sono contento dell'acquisto", "pos"),
    ("assistenza gentile e prezzo giusto", "pos"),
    ("un'esperienza fantastica, lo consiglio", "pos"),
    ("il prodotto è pessimo e molto caro", "neg"),
    ("ho odiato il servizio, un vero disastro", "neg"),
    ("la consegna è stata lenta e la spedizione è arrivata rotta", "neg"),
    ("scarsa qualità, sono deluso dell'acquisto", "neg"),
    ("assistenza orribile e prezzo esagerato", "neg"),
    ("un'esperienza terribile, lo sconsiglio", "neg"),
]
PRUEBA_IT = [
    ("un prodotto eccellente, tutto perfetto", "pos"),
    ("l'assistenza è stata gentile e la spedizione veloce", "pos"),
    ("la spedizione non è stata lenta", "pos"),
    ("un acquisto geniale", "pos"),
    ("un disastro, tutto molto caro", "neg"),
    ("la qualità non è buona", "neg"),
    ("un acquisto scadente", "neg"),
    ("che meraviglia, si è rotto il primo giorno!", "neg"),
]
POSITIVAS_ES = {"excelente", "útil", "encantó", "perfecto", "rápida", "rápido", "bien", "buena", "bueno",
                "contento", "amable", "justo", "fantástica", "recomiendo", "genial", "maravilla", "gustó"}
NEGATIVAS_ES = {"malo", "caro", "odié", "desastre", "lenta", "lento", "roto", "pésima", "decepcionado",
                "horrible", "abusivo", "terrible", "desaconsejo", "lamentable"}
POSITIVAS_IT = {"eccellente", "utile", "piaciuto", "perfetto", "veloce", "bene", "buona", "buono", "ottima",
                "contento", "gentile", "giusto", "fantastica", "consiglio", "geniale", "meraviglia"}
NEGATIVAS_IT = {"pessimo", "caro", "odiato", "disastro", "lenta", "lento", "rotta", "rotto", "scarsa", "deluso",
                "orribile", "esagerato", "terribile", "sconsiglio", "scadente"}
NEGADORES_ES = {"no", "nunca", "ni", "tampoco"}
NEGADORES_IT = {"non", "mai", "né"}

In [ ]:
def tokenizar(texto):
    return re.findall(r"\w+", texto.lower())

def clasificar_reglas(texto, positivas, negativas, negadores=None, ventana=3):
    palabras = tokenizar(texto)
    puntos_pos = puntos_neg = 0
    for i, palabra in enumerate(palabras):
        valor = 1 if palabra in positivas else -1 if palabra in negativas else 0
        if valor and negadores and any(x in negadores for x in palabras[max(0, i - ventana):i]):
            valor = -valor                       # «no es bueno»: la negación invierte la polaridad
        puntos_pos += valor > 0
        puntos_neg += valor < 0
    return "pos" if puntos_pos > puntos_neg else "neg" if puntos_neg > puntos_pos else None

def entrenar_conteos(ejemplos):
    conteos = {"pos": Counter(), "neg": Counter()}
    for texto, etiqueta in ejemplos:
        conteos[etiqueta].update(tokenizar(texto))
    return conteos

def clasificar_conteos(texto, conteos):
    peso_pos = sum(conteos["pos"][p] for p in tokenizar(texto))
    peso_neg = sum(conteos["neg"][p] for p in tokenizar(texto))
    return "pos" if peso_pos > peso_neg else "neg" if peso_neg > peso_pos else None

def exactitud(clasificador, ejemplos):
    return sum(clasificador(texto) == real for texto, real in ejemplos) / len(ejemplos)

DATOS = {
    "spa": (ENTRENAMIENTO_ES, PRUEBA_ES, POSITIVAS_ES, NEGATIVAS_ES, NEGADORES_ES),
    "ita": (ENTRENAMIENTO_IT, PRUEBA_IT, POSITIVAS_IT, NEGATIVAS_IT, NEGADORES_IT),
}
print("Balance del entrenamiento:", {i: Counter(e for _, e in d[0]) for i, d in DATOS.items()})
print("Balance de la prueba:", {i: Counter(e for _, e in d[1]) for i, d in DATOS.items()})

In [ ]:
clasificadores = {}
resultados = {}
for idioma, (entren, prueba, pos, neg, nega) in DATOS.items():
    conteos = entrenar_conteos(entren)
    clasificadores[idioma] = {
        "reglas": lambda t, pos=pos, neg=neg: clasificar_reglas(t, pos, neg),
        "reglas + negación": lambda t, pos=pos, neg=neg, nega=nega: clasificar_reglas(t, pos, neg, nega),
        "recuentos": lambda t, conteos=conteos: clasificar_conteos(t, conteos),
    }
    resultados[idioma] = {nombre: exactitud(f, prueba) for nombre, f in clasificadores[idioma].items()}

tabla = pd.DataFrame(resultados).rename(columns={"spa": "español", "ita": "italiano"})
print((tabla * 100).round(1).astype(str) + " %")

In [ ]:
nombres = list(resultados["spa"].keys())
ancho = 0.35
posiciones = np.arange(len(nombres))
plt.bar(posiciones - ancho / 2, [resultados["spa"][n] for n in nombres], ancho, label="español", color="#c0392b")
plt.bar(posiciones + ancho / 2, [resultados["ita"][n] for n in nombres], ancho, label="italiano", color="#27ae60")
plt.xticks(posiciones, nombres)
plt.ylim(0, 1)
plt.ylabel("Exactitud sobre 8 frases")
plt.title("Reglas frente a recuentos (ejemplo ilustrativo, 8 frases de prueba)")
plt.legend()
plt.show()

**¿Qué observas?** Antes de mirar los porcentajes, mira **cómo falla** cada clasificador. Veamos los errores frase a frase.

In [ ]:
for idioma, (entren, prueba, pos, neg, nega) in DATOS.items():
    print("=" * 8, idioma)
    for nombre, f in clasificadores[idioma].items():
        errores = [(texto, real, f(texto) or "—") for texto, real in prueba if f(texto) != real]
        print(f"Clasificador '{nombre}': {len(errores)} errores")
        for texto, real, predicho in errores:
            print(f"    {texto!r}  real={real}  predicho={predicho}")

In [ ]:
for idioma, (entren, prueba, pos, neg, nega) in DATOS.items():
    vocabulario = {p for texto, _ in entren for p in tokenizar(texto)}
    palabras_prueba = [p for texto, _ in prueba for p in tokenizar(texto)]
    fuera = [p for p in palabras_prueba if p not in vocabulario]
    print(f"{idioma}: {len(fuera)} de {len(palabras_prueba)} palabras de prueba no aparecen en el entrenamiento")

**¿Qué observas?**

- Las **reglas sin negación** fallan donde hay un `no`/`non` delante de una palabra con carga (`el envío no fue lento`, `la calidad no es buena`); con negación esos casos se arreglan.
- Los **recuentos** fallan por **palabras que nunca vieron** (`genial`, `lamentable`, `geniale`, `scadente`): con solo 12 frases de entrenamiento, buena parte del vocabulario de prueba es nuevo. También fallan con la negación, porque `no` y `lento` no están enlazados en una bolsa de palabras.
- **Ninguno** de los tres entiende la ironía de `¡qué maravilla, se rompió al primer día!`.
- Un solo número de exactitud oculta todo esto. Por eso miramos los errores, y por eso el 62,5 % o el 87,5 % de aquí son **ilustrativos**: no se pueden generalizar a otros datos. En la clase 5 aprenderás a evaluar con datos suficientes y con la métrica adecuada.

---
## 7. Resumen y siguiente paso

- El NLP convierte texto en datos: casi toda tarea se escribe como **entrada → salida** (clasificar, extraer, transformar, generar, recuperar, analizar).
- La **pipeline clásica** (adquisición → limpieza → tokenización → representación → modelo → evaluación) se elige según la tarea y rara vez se recorre una sola vez.
- El lenguaje es difícil por la **ambigüedad** (aquí, léxica y sintáctica); el modelo «elige», no «entiende». Referencia, contexto y variación se estudian en la clase 2.
- El italiano y el español tienen **morfología rica**: 38 y 37 formas distintas para un verbo regular frente a 5 del inglés, clíticos y concordancia de género y número. Las herramientas se adaptan por idioma.
- **Clásico y neuronal** se comparan con criterios (datos, coste, explicabilidad, robustez, recursos, mantenimiento); en la clase 12 se comparan con cifras.
- Un porcentaje sobre 8 frases es solo **ilustrativo**; hay que mirar los errores.

**Siguiente:** en la clase 2, *Texto bruto y tokenización*, arreglarás de verdad el paso 3 de la pipeline: Unicode, expresiones regulares y tokenización. Antes, haz el notebook de práctica de esta clase.